# Hospital Appointment No-Show Prediction: EDA & Modeling

**Objective:** Predict outpatient appointment no-shows (`no_show = 1` when `No-show == "Yes"`) so clinic staff can prioritize proactive reminder calls for the **top 20% highest-risk appointments** — never to deny or penalize care.

Key engineering safeguards in this notebook:
- **Primary evaluation:** Chronological **time-based split** on `AppointmentDay` (`< 2016-06-01` train, `>= 2016-06-01` test).
- **Leakage guards:** Strictly past-only patient history (`< AppointmentDay`), `Neighbourhood` target/frequency encoding fitted inside the training pipeline only, and `PatientId`/`AppointmentID` excluded from model features.
- **Metrics:** PR-AUC, ROC-AUC, Top-10%/20% Lift & Recall, Brier calibration — never leading with accuracy (~80% base rate).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_prep import clean_appointments, load_raw_data
from src.features import engineer_features, get_feature_matrix_and_target
from src.evaluate import compute_binary_metrics, compute_subgroup_metrics
from src.train import make_logreg_pipeline, make_hgb_pipeline, time_based_split

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)
SEED = 42

## 1. Data Loading, Known Pitfalls & Cleaning Audit

In [2]:
df_raw, data_source = load_raw_data(PROJECT_ROOT / 'data' / 'raw' / 'KaggleV2-May-2016.csv')
print(f'Data source: {data_source} | Raw shape: {df_raw.shape}')
df_clean, audit = clean_appointments(df_raw, data_source=data_source)
print(json.dumps(audit, indent=2))

Data source: real | Raw shape: (110527, 14)


{
  "data_source": "real",
  "raw_rows": 110527,
  "steps": [
    {
      "step": "deduplicate_appointment_id",
      "rows_removed": 0,
      "rows_remaining": 110527
    },
    {
      "step": "drop_negative_lead_time",
      "rows_removed": 5,
      "rows_remaining": 110522,
      "reason": "AppointmentDay strictly before ScheduledDay"
    },
    {
      "step": "drop_invalid_age",
      "rows_removed": 6,
      "negative_age_rows": 1,
      "extreme_age_rows_gt_110": 5,
      "rows_remaining": 110516,
      "reason": "Keep Age in [0, 110]; Age == 0 retained as valid infant visits"
    }
  ],
  "clean_rows": 110516,
  "total_rows_removed": 11,
  "no_show_rate": 0.20188,
  "unique_patients": 62296,
  "unique_neighbourhoods": 81,
  "appointment_date_min": "2016-04-29",
  "appointment_date_max": "2016-06-08"
}


## 2. Leakage-Free Feature Engineering & Exploratory Data Analysis

We compute strictly past-only patient history (`< appointment_date`) and static appointment features, then inspect six key patterns across the cleaned dataset.

In [3]:
df_feat = engineer_features(df_clean)
base_rate = df_feat['no_show'].mean()
print(f'Clean rows: {len(df_feat):,} | Overall no-show rate: {base_rate:.2%}')

Clean rows: 110,516 | Overall no-show rate: 20.19%


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# 1. Lead time bucket
lead_order = ['same_day', '1_to_3_days', '4_to_7_days', '8_to_14_days', '15_to_30_days', '31_plus_days']
s1 = df_feat.groupby('lead_time_bucket')['no_show'].agg(['mean', 'count']).reindex(lead_order)
sns.barplot(x=s1.index, y=s1['mean'], ax=axes[0, 0], color='#2b6cb0')
axes[0, 0].axhline(base_rate, color='crimson', ls='--')
axes[0, 0].set_title('1. No-Show by Lead Time Bucket')
axes[0, 0].tick_params(axis='x', rotation=25)

# 2. Age band
age_order = ['0_12', '13_17', '18_29', '30_44', '45_59', '60_74', '75_plus']
s2 = df_feat.groupby('age_band')['no_show'].agg(['mean', 'count']).reindex(age_order)
sns.barplot(x=s2.index, y=s2['mean'], ax=axes[0, 1], color='#2f855a')
axes[0, 1].axhline(base_rate, color='crimson', ls='--')
axes[0, 1].set_title('2. No-Show by Age Band')

# 3. Weekday
day_order = [d for d in ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat'] if d in df_feat['appt_weekday'].unique()]
s3 = df_feat.groupby('appt_weekday')['no_show'].agg(['mean', 'count']).reindex(day_order)
sns.barplot(x=s3.index, y=s3['mean'], ax=axes[0, 2], color='#4a5568')
axes[0, 2].axhline(base_rate, color='crimson', ls='--')
axes[0, 2].set_title('3. No-Show by Weekday')

# 4. SMS confounding (stratified by lead > 0)
s4 = (
    df_feat.assign(lead_grp=np.where(df_feat['lead_time_days'] == 0, 'Same-day (0d)', 'Advance (>0d)'))
    .groupby(['lead_grp', 'sms_received'])['no_show'].mean().reset_index()
)
sns.barplot(data=s4, x='lead_grp', y='no_show', hue='sms_received', ax=axes[1, 0])
axes[1, 0].set_title('4. SMS Confounding (Stratified by Lead Time)')

# 5. Chronic conditions & Scholarship
cond_cols = ['hypertension', 'diabetes', 'alcoholism', 'has_disability', 'scholarship']
cond_rates = {c: df_feat.loc[df_feat[c] == 1, 'no_show'].mean() for c in cond_cols}
sns.barplot(x=list(cond_rates.keys()), y=list(cond_rates.values()), ax=axes[1, 1], color='#805ad5')
axes[1, 1].axhline(base_rate, color='crimson', ls='--')
axes[1, 1].set_title('5. No-Show Rate When Condition = 1')
axes[1, 1].tick_params(axis='x', rotation=20)

# 6. Top 5 & Bottom 5 Neighbourhoods (n >= 200)
hs = df_feat.groupby('neighbourhood')['no_show'].agg(['mean', 'count'])
hs = hs[hs['count'] >= 200].sort_values('mean')
tb = pd.concat([hs.head(5), hs.tail(5)])
axes[1, 2].barh(tb.index, tb['mean'], color=['#38a169']*5 + ['#e53e3e']*5)
axes[1, 2].axvline(base_rate, color='black', ls='--')
axes[1, 2].set_title('6. Top/Bottom Neighbourhoods (n >= 200)')

plt.tight_layout()
plt.show()

C:\Users\hp\AppData\Local\Temp\ipykernel_25560\3838431007.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Key EDA Takeaways:**
1. **Lead time is the strongest driver:** Same-day visits miss only ~4.7% of the time vs. >28–33% for visits scheduled 15+ days ahead.
2. **Age effect:** Teens and young adults (13–29) miss at the highest rates; older adults (60–74) attend most reliably.
3. **Weekday:** Modest Monday–Friday variation (~19–21%), while Saturday has a tiny sample size (`n=39`).
4. **SMS confounding:** Same-day appointments (`lead == 0`) receive zero SMS reminders. Among advance appointments (`lead > 0`), SMS-sent visits have a *lower* no-show rate (~27.6% vs. ~29.4%).
5. **Conditions & Scholarship:** Hypertension/diabetes patients attend more regularly (older age/ongoing care), while Scholarship recipients have higher no-show rates (~23.7% vs. ~19.8%), reflecting socioeconomic barriers.
6. **Neighbourhoods:** Meaningful geographic variation across neighbourhoods with `n >= 200`, motivating leakage-safe target/frequency encoding inside CV folds.

## 3. Time-Based Modeling & Comparison Against Baselines

In [5]:
df_train, df_test, cutoff = time_based_split(df_feat)
X_train, y_train = get_feature_matrix_and_target(df_train)
X_test, y_test = get_feature_matrix_and_target(df_test)
print(f'Time cutoff: {cutoff} | Train: {len(df_train):,} ({y_train.mean():.2%}) | Test: {len(df_test):,} ({y_test.mean():.2%})')

logreg = make_logreg_pipeline().fit(X_train, y_train)
p_log = logreg.predict_proba(X_test)[:, 1]
t_log = float(np.quantile(logreg.predict_proba(X_train)[:, 1], 0.80))

hgb = make_hgb_pipeline({'max_iter': 250, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 100}).fit(X_train, y_train)
p_hgb = hgb.predict_proba(X_test)[:, 1]
t_hgb = float(np.quantile(hgb.predict_proba(X_train)[:, 1], 0.80))

p_base_a = np.zeros(len(y_test))
p_base_b = (df_test['lead_time_days'].to_numpy() >= 14).astype(float)

table = pd.DataFrame({
    'Always Attends': compute_binary_metrics(y_test, p_base_a, np.zeros(len(y_test), dtype=int), 'Always 0'),
    'Rule (lead >= 14d)': compute_binary_metrics(y_test, p_base_b, p_base_b.astype(int), 'lead >= 14'),
    'LogisticRegression': compute_binary_metrics(y_test, p_log, (p_log >= t_log).astype(int), f'p >= {t_log:.3f}'),
    'HistGradientBoosting': compute_binary_metrics(y_test, p_hgb, (p_hgb >= t_hgb).astype(int), f'p >= {t_hgb:.3f}'),
}).T
table

Time cutoff: 2016-06-01 | Train: 84,067 (20.73%) | Test: 26,449 (18.46%)


,threshold_rule,flagged_rate,roc_auc,pr_auc,brier_score,precision,recall,f1,accuracy_reference_only,top10_precision,top10_recall,top10_lift,top20_precision,top20_recall,top20_lift
Always Attends,Always 0,0.0,0.5,0.1846,0.1846,0.0,0.0,0.0,0.8154,0.1846,0.1,1.0,0.1846,0.2,1.0
Rule (lead >= 14d),lead >= 14,0.3031,0.6065,0.235,0.3117,0.2903,0.4769,0.3609,0.6883,0.2903,0.1573,1.573,0.2903,0.3146,1.573
LogisticRegression,p >= 0.655,0.1901,0.7328,0.3454,0.2145,0.3524,0.363,0.3576,0.7593,0.3905,0.2116,2.1159,0.3488,0.3779,1.8895
HistGradientBoosting,p >= 0.332,0.1834,0.7345,0.3479,0.1355,0.3553,0.3531,0.3542,0.7623,0.3924,0.2126,2.1261,0.349,0.3781,1.8905


## 4. Saved Full Pipeline Results (Calibration + GroupKFold Robustness Check)

Below we inspect `models/metrics.json` produced by `python -m src.train`, which also includes OOF-derived top-20% thresholds, isotonic calibration, 5-fold `GroupKFold` by `PatientId`, and subgroup fairness metrics.

In [6]:
metrics_file = PROJECT_ROOT / 'models' / 'metrics.json'
if metrics_file.exists():
    with open(metrics_file, 'r', encoding='utf-8') as f:
        m = json.load(f)
    print('Selected deployed model:', m['selected_model'])
    print('\n--- Full Test Comparison ---')
    display(pd.DataFrame(m['test_metrics']).T)
    print('\n--- 5-Fold GroupKFold (PatientId) Robustness Check ---')
    display(pd.DataFrame(m['group_kfold_cv']).T)
else:
    print('Run `python -m src.train` to generate models/metrics.json.')

Selected deployed model: HistGradientBoosting (isotonic cal.)

--- Full Test Comparison ---


,threshold_rule,flagged_rate,roc_auc,pr_auc,brier_score,precision,recall,f1,accuracy_reference_only,top10_precision,top10_recall,top10_lift,top20_precision,top20_recall,top20_lift
Baseline A: Always Attends,Always 0,0.0,0.5,0.1846,0.1846,0.0,0.0,0.0,0.8154,0.1846,0.1,1.0,0.1846,0.2,1.0
Baseline B: Rule (lead_time >= 14d),lead_time >= 14,0.3031,0.6065,0.235,0.3117,0.2903,0.4769,0.3609,0.6883,0.2903,0.1573,1.573,0.2903,0.3146,1.573
Logistic Regression (balanced),Top-20% OOF p>=0.656,0.19,0.7328,0.3454,0.2145,0.3527,0.363,0.3578,0.7595,0.3905,0.2116,2.1159,0.3488,0.3779,1.8895
HistGradientBoosting (tuned),Top-20% OOF p>=0.330,0.1873,0.7356,0.35,0.1352,0.3603,0.3656,0.363,0.7631,0.3955,0.2143,2.1425,0.3578,0.3878,1.9387
HistGradientBoosting (isotonic cal.),Top-20% OOF p>=0.325,0.1831,0.7368,0.35,0.1352,0.3655,0.3626,0.364,0.7662,0.3981,0.2157,2.1566,0.3634,0.3938,1.969



--- 5-Fold GroupKFold (PatientId) Robustness Check ---


,n_splits,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,brier_mean
Logistic Regression (balanced),5.0,0.7339,0.0018,0.3691,0.0045,0.2120
HistGradientBoosting,5.0,0.7390,0.0011,0.3789,0.0071,0.1426
